# Aerial Human Detection — Batch Folder Inference + ByteTrack V2
## Google Colab — Direct Google Drive model links supported

This version accepts **Google Drive shared FILE links for all three final model checkpoints**, in addition to mounted Drive paths. It also processes a whole folder of images/videos with YOLO26s, RT-DETR-R18 and BPD-YOLOn/L-FPN.

**Images:** detection only.  
**Videos:** detection + ByteTrack.  
**RUN_MODEL = "ALL":** every media file is processed sequentially by all three models.


## 1. Mount Google Drive

In [1]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
print("Google Drive mounted.")

Mounted at /content/drive
Google Drive mounted.


## 2. Install dependencies

In [2]:
import subprocess
import sys

packages = [
    "ultralytics==8.4.116",
    "PyYAML>=6.0",
    "pandas>=2.0",
    "tqdm>=4.66",
    "pillow>=10.0",
    "gdown>=5.2.0",
    "faster-coco-eval>=1.6.7",
]

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade-strategy",
        "only-if-needed",
        *packages,
    ]
)

import cv2
import numpy as np
import pandas as pd
import torch
import torchvision
import ultralytics
import gdown

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Torchvision :", torchvision.__version__)
print("CUDA        :", torch.version.cuda)
print("Ultralytics :", ultralytics.__version__)
print("OpenCV      :", cv2.__version__)
print("gdown       :", gdown.__version__)

if not torch.cuda.is_available():
    print("WARNING: CUDA is not available. Inference will run on CPU.")
else:
    print("GPU         :", torch.cuda.get_device_name(0))

print("Runtime setup: PASSED")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Python      : 3.12.13
PyTorch     : 2.11.0+cu128
Torchvision : 0.26.0+cu128
CUDA        : 12.8
Ultralytics : 8.4.116
OpenCV      : 5.0.0
gdown       : 5.2.2
GPU         : NVIDIA L4
Runtime setup: PASSED


## 3. USER CONFIGURATION

### How to use
Choose one input mode:

#### Mode A — local folder already inside your Google Drive
```python
INPUT_SOURCE_MODE = "LOCAL_FOLDER"
LOCAL_INPUT_PATH = "/content/drive/MyDrive/your_folder"
```

#### Mode B — Google Drive shared folder link
```python
INPUT_SOURCE_MODE = "GDRIVE_SHARED_FOLDER"
GDRIVE_SHARED_URL = "https://drive.google.com/drive/folders/..."
```

#### Mode C — Google Drive shared single file link
```python
INPUT_SOURCE_MODE = "GDRIVE_SHARED_FILE"
GDRIVE_SHARED_URL = "https://drive.google.com/file/d/.../view?usp=drive_link"
```

### Model execution
Set:
- one model, or
- `"ALL"` for all three models.

In [3]:
from pathlib import Path

# ============================================================
# FINAL MODEL SOURCES
#
# Each source can be EITHER:
# 1) a mounted Google Drive path:
#    /content/drive/MyDrive/.../best.pt
#
# OR
#
# 2) a Google Drive shared FILE URL:
#    https://drive.google.com/file/d/.../view?usp=drive_link
#
# IMPORTANT:
# Do NOT wrap a Google Drive URL in Path(...).
# Keep shared links as plain strings.
# ============================================================

YOLO26S_MODEL_SOURCE = (
    "https://drive.google.com/file/d/"
    "1Ds2wuTMp1voGOH-VbNhqFfM-xwhpvBcL/"
    "view?usp=drive_link"
)

RTDETR_MODEL_SOURCE = (
    "https://drive.google.com/file/d/"
    "1Ku_Sh9i-aCN2aLaDQH8tDfgrTzBFzZ3M/"
    "view?usp=drive_link"
)

BPD_MODEL_SOURCE = (
    "https://drive.google.com/file/d/"
    "1sV5nb8uU4E_b_7JDitpbYbxoM0x2ob2H/"
    "view?usp=drive_link"
)


# ============================================================
# INPUT SOURCE
# ============================================================

INPUT_SOURCE_MODE = "LOCAL_FOLDER"

# INPUT_SOURCE_MODE = "GDRIVE_SHARED_FOLDER"
# INPUT_SOURCE_MODE = "GDRIVE_SHARED_FILE"

LOCAL_INPUT_PATH = Path(
    "/content/drive/MyDrive/referee_videos"
)

GDRIVE_SHARED_URL = ""


# ============================================================
# MODEL SELECTION
# ============================================================

RUN_MODEL = "ALL"

# RUN_MODEL = "YOLO26s"
# RUN_MODEL = "RT-DETR-R18"
# RUN_MODEL = "BPD-YOLOn/L-FPN"


# ============================================================
# OUTPUT ROOT
# ============================================================

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/"
    "aerial_human_detection/"
    "batch_inference_outputs_L4_V2"
)


# ============================================================
# COMMON DETECTION SETTINGS
# ============================================================

# High-resolution inference is important for small aerial persons.
IMAGE_SIZE = 1280

# Keep the detector reasonably sensitive to small persons,
# but do not lower it too much because false detections would
# also be passed to ByteTrack.
CONF_THRESHOLD = 0.08

# NMS / prediction IoU threshold.
IOU_THRESHOLD = 0.70

# Dense aerial scenes may contain many persons.
MAX_DETECTIONS = 3000

# Only class 0 = person is used by the detector backends.
PERSON_CLASS_ID = 0

DEVICE = (
    "cuda:0"
    if __import__("torch").cuda.is_available()
    else "cpu"
)

# Keep reference PyTorch inference in FP32.
USE_FP16 = False


# ============================================================
# BYTETRACK SETTINGS
# ROBUST AERIAL PERSON TRACKING PROFILE
# ============================================================

# First-stage association:
# Only reasonably confident detections are used for the
# strongest track association.
TRACK_HIGH_THRESH = 0.30

# Second-stage association:
# Keep lower-confidence person detections available for
# reconnecting an already existing track.
#
# This should not be lower than the detector confidence floor
# unless the detector itself also outputs detections below it.
TRACK_LOW_THRESH = 0.08

# IMPORTANT:
# A new ID is created only when an unmatched detection has
# sufficiently high confidence.
#
# Raising this value reduces false/new IDs generated from
# weak or noisy detections.
NEW_TRACK_THRESH = 0.40

# Keep lost tracks alive longer.
# This helps preserve the same ID after short occlusions,
# blur, camera motion, or temporary missed detections.
TRACK_BUFFER = 60

# Slightly more tolerant association than the default 0.80.
# This can help reconnect the same person across adjacent
# frames without making matching excessively permissive.
MATCH_THRESH = 0.82

# Fuse detection confidence with spatial association.
FUSE_SCORE = True


# ============================================================
# TRACK VISUALIZATION SETTINGS
# ============================================================

SHOW_BOXES = True
SHOW_TRACK_ID = True
SHOW_CONFIDENCE = True
SHOW_TRAILS = True

# Longer history makes identity continuity easier to inspect.
TRAIL_LENGTH = 60


# ============================================================
# OUTPUT SETTINGS
# ============================================================

SAVE_VIDEO = True
SAVE_CSV = True
SAVE_JSON = True

# Process every frame.
VIDEO_FRAME_STRIDE = 1


# ============================================================
# DRAWING SETTINGS
# ============================================================

BOX_THICKNESS = 2
TEXT_SCALE = 0.55
TEXT_THICKNESS = 1


# ============================================================
# PERFORMANCE SETTINGS
# ============================================================

WARMUP_RUNS = 2

# FPS smoothing only affects displayed FPS.
FPS_SMOOTHING_WINDOW = 30


# ============================================================
# INPUT FOLDER SCAN SETTINGS
# ============================================================

# Search inside subfolders too.
RECURSIVE_SCAN = True

# None = process every supported image/video.
LIMIT_MEDIA_COUNT = None

# Example for testing:
# LIMIT_MEDIA_COUNT = 2


# ============================================================
# PINNED RT-DETR SOURCE
# ============================================================

RTDETR_REPO_URL = (
    "https://github.com/lyuwenyu/RT-DETR.git"
)

RTDETR_REPO_COMMIT = (
    "199fc382f53abbfb5c1804c97b0e8b204e3cb8d0"
)


# ============================================================
# STEP-3 VALIDATION METRICS
#
# These are reference validation results from Step 3.
# They are NOT accuracy values measured on the current video.
# ============================================================

STEP3_VALIDATION = {

    "YOLO26s": {
        "map50_95": 0.31060,
        "ap50": 0.67438,
        "best_epoch": 30,
    },

    "RT-DETR-R18": {
        "map50_95": 0.34854,
        "ap50": 0.70839,
        "best_epoch": 45,
    },

    "BPD-YOLOn/L-FPN": {
        "map50_95": 0.29366,
        "ap50": 0.65557,
        "best_epoch": 43,
    },
}


# ============================================================
# CREATE OUTPUT DIRECTORY
# ============================================================

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# CONFIGURATION SUMMARY
# ============================================================

print("=" * 72)
print("AERIAL PERSON DETECTION + BYTETRACK CONFIGURATION")
print("=" * 72)

print("Input mode          :", INPUT_SOURCE_MODE)
print("Input folder        :", LOCAL_INPUT_PATH)
print("Run model           :", RUN_MODEL)
print("Device              :", DEVICE)
print("Image size          :", IMAGE_SIZE)

print()
print("Detection settings")
print("------------------")
print("Person class ID     :", PERSON_CLASS_ID)
print("Confidence          :", CONF_THRESHOLD)
print("IoU                 :", IOU_THRESHOLD)
print("Maximum detections  :", MAX_DETECTIONS)

print()
print("ByteTrack settings")
print("------------------")
print("Track high threshold:", TRACK_HIGH_THRESH)
print("Track low threshold :", TRACK_LOW_THRESH)
print("New track threshold :", NEW_TRACK_THRESH)
print("Track buffer        :", TRACK_BUFFER)
print("Match threshold     :", MATCH_THRESH)
print("Fuse score          :", FUSE_SCORE)

print()
print("Output")
print("------")
print("Output root         :", OUTPUT_ROOT)

print("=" * 72)
print("Configuration: READY")
print("=" * 72)

AERIAL PERSON DETECTION + BYTETRACK CONFIGURATION
Input mode          : LOCAL_FOLDER
Input folder        : /content/drive/MyDrive/referee_videos
Run model           : ALL
Device              : cuda:0
Image size          : 1280

Detection settings
------------------
Person class ID     : 0
Confidence          : 0.08
IoU                 : 0.7
Maximum detections  : 3000

ByteTrack settings
------------------
Track high threshold: 0.3
Track low threshold : 0.08
New track threshold : 0.4
Track buffer        : 60
Match threshold     : 0.82
Fuse score          : True

Output
------
Output root         : /content/drive/MyDrive/aerial_human_detection/batch_inference_outputs_L4_V2
Configuration: READY


## 4. Imports, utilities and drawing

In [4]:
from __future__ import annotations

import contextlib
import gc
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time

from collections import defaultdict, deque
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace
from typing import Optional

import cv2
import numpy as np
import pandas as pd
import torch

from PIL import Image
from IPython.display import display, Video
from tqdm.auto import tqdm


IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"
}

VIDEO_EXTENSIONS = {
    ".mp4", ".mov", ".avi", ".mkv", ".m4v", ".webm", ".wmv", ".mpeg", ".mpg"
}


@dataclass
class DetectionBatch:
    boxes: np.ndarray
    scores: np.ndarray
    classes: np.ndarray
    detector_ms: float

    @classmethod
    def empty(cls, detector_ms: float = 0.0):
        return cls(
            boxes=np.zeros((0, 4), dtype=np.float32),
            scores=np.zeros((0,), dtype=np.float32),
            classes=np.zeros((0,), dtype=np.float32),
            detector_ms=float(detector_ms),
        )

    def __len__(self):
        return len(self.boxes)


def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while True:
            block = stream.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sanitize_name(value: str) -> str:
    value = value.replace("/", "_").replace("\\", "_")
    return "".join(ch if ch.isalnum() or ch in {"-", "_", "."} else "_" for ch in value)


def classify_media(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix in IMAGE_EXTENSIONS:
        return "image"
    if suffix in VIDEO_EXTENSIONS:
        return "video"

    image = cv2.imread(str(path))
    if image is not None:
        return "image"

    cap = cv2.VideoCapture(str(path))
    ok = cap.isOpened()
    cap.release()
    if ok:
        return "video"

    raise ValueError(f"Could not classify media: {path}")


def is_supported_media(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in (IMAGE_EXTENSIONS | VIDEO_EXTENSIONS)


def deterministic_track_color(track_id: int):
    hue = int((track_id * 47) % 180)
    hsv = np.uint8([[[hue, 210, 255]]])
    bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)[0, 0]
    return tuple(int(x) for x in bgr.tolist())


def draw_text_box(
    image,
    text,
    origin,
    bg_color=(20, 20, 20),
    fg_color=(255, 255, 255),
    alpha=0.75,
    scale=0.55,
    thickness=1,
):
    x, y = origin
    (tw, th), baseline = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, thickness)
    x1 = max(0, x)
    y1 = max(0, y - th - baseline - 6)
    x2 = min(image.shape[1] - 1, x + tw + 8)
    y2 = min(image.shape[0] - 1, y + 2)

    overlay = image.copy()
    cv2.rectangle(overlay, (x1, y1), (x2, y2), bg_color, -1)
    cv2.addWeighted(overlay, alpha, image, 1.0 - alpha, 0, image)

    cv2.putText(
        image,
        text,
        (x1 + 4, y - baseline - 2),
        cv2.FONT_HERSHEY_SIMPLEX,
        scale,
        fg_color,
        thickness,
        cv2.LINE_AA,
    )


def draw_detection_boxes(frame, boxes, scores, track_ids=None, trails=None):
    output = frame.copy()

    if track_ids is None:
        track_ids = [None] * len(boxes)

    for box, score, track_id in zip(boxes, scores, track_ids):
        x1, y1, x2, y2 = [int(round(v)) for v in box]

        color = deterministic_track_color(int(track_id)) if track_id is not None else (0, 200, 255)

        if SHOW_BOXES:
            cv2.rectangle(output, (x1, y1), (x2, y2), color, BOX_THICKNESS)

        label_parts = ["person"]
        if SHOW_TRACK_ID and track_id is not None:
            label_parts.append(f"ID {int(track_id)}")
        if SHOW_CONFIDENCE:
            label_parts.append(f"{float(score):.2f}")

        draw_text_box(
            output,
            " | ".join(label_parts),
            (x1, max(18, y1)),
            bg_color=color,
            fg_color=(255, 255, 255),
            alpha=0.78,
            scale=TEXT_SCALE,
            thickness=TEXT_THICKNESS,
        )

    if SHOW_TRAILS and trails:
        for track_id, points in trails.items():
            if len(points) < 2:
                continue
            color = deterministic_track_color(int(track_id))
            pts = np.asarray(points, dtype=np.int32).reshape(-1, 1, 2)
            cv2.polylines(output, [pts], False, color, 2, cv2.LINE_AA)

    return output


def add_video_status_overlay(
    frame,
    model_name,
    frame_number,
    source_fps,
    processing_fps,
    detector_ms,
    total_ms,
    detections,
    active_tracks,
    unique_tracks,
    mean_conf,
):
    output = frame.copy()

    panel_h = min(118, max(92, int(output.shape[0] * 0.11)))
    overlay = output.copy()

    cv2.rectangle(overlay, (0, 0), (output.shape[1], panel_h), (10, 16, 24), -1)
    cv2.addWeighted(overlay, 0.78, output, 0.22, 0, output)

    val_metrics = STEP3_VALIDATION[model_name]

    line1 = (
        f"{model_name} | person | ByteTrack | "
        f"Frame {frame_number} | Source {source_fps:.2f} FPS"
    )
    line2 = (
        f"Processing {processing_fps:.2f} FPS | "
        f"Detector {detector_ms:.1f} ms | "
        f"Total {total_ms:.1f} ms | "
        f"Det {detections} | Active IDs {active_tracks} | "
        f"Unique IDs {unique_tracks}"
    )
    line3 = (
        f"Mean Det Conf {mean_conf:.3f} | "
        f"Step3 Val mAP50-95 {val_metrics['map50_95']:.5f} | "
        f"AP50 {val_metrics['ap50']:.5f} | "
        f"imgsz {IMAGE_SIZE} | conf {CONF_THRESHOLD:.2f}"
    )

    lines = [line1, line2, line3]
    y = 28

    for line in lines:
        cv2.putText(output, line, (14, y), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (255, 255, 255), 1, cv2.LINE_AA)
        y += 30

    return output


def build_image_footer(annotated, model_name, model_path, detector_ms, detections, mean_conf):
    height, width = annotated.shape[:2]
    footer_h = 190
    footer = np.full((footer_h, width, 3), (22, 28, 36), dtype=np.uint8)

    val_metrics = STEP3_VALIDATION[model_name]

    fps = 1000.0 / detector_ms if detector_ms > 0 else 0.0

    lines = [
        f"Model: {model_name} | Class: person | Checkpoint: {Path(model_path).name}",
        f"Input profile: imgsz={IMAGE_SIZE} | conf={CONF_THRESHOLD:.2f} | IoU={IOU_THRESHOLD:.2f} | max_det={MAX_DETECTIONS}",
        f"Detections: {detections} | Mean Det Conf: {mean_conf:.3f} | Detector latency: {detector_ms:.1f} ms | Throughput: {fps:.2f} FPS",
        f"Step3 VisDrone validation: mAP50-95={val_metrics['map50_95']:.5f} | AP50={val_metrics['ap50']:.5f} | Best epoch={val_metrics['best_epoch']}",
        f"Device: {DEVICE} | FP16: {USE_FP16} | Note: confidence is not ground-truth accuracy.",
    ]

    y = 34
    for line in lines:
        cv2.putText(footer, line, (18, y), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (245, 245, 245), 1, cv2.LINE_AA)
        y += 32

    return np.vstack([annotated, footer])


def maybe_transcode_h264(source_path: Path) -> Path:
    if not shutil.which("ffmpeg"):
        return source_path

    target = source_path.with_name(source_path.stem + "_h264.mp4")

    command = [
        "ffmpeg",
        "-y",
        "-loglevel",
        "error",
        "-i",
        str(source_path),
        "-c:v",
        "libx264",
        "-preset",
        "medium",
        "-crf",
        "20",
        "-pix_fmt",
        "yuv420p",
        "-an",
        str(target),
    ]

    result = subprocess.run(command, check=False)

    if result.returncode == 0 and target.exists():
        return target

    return source_path

## 5. Resolve the input source (whole folder or shared link)

In [5]:
import gdown

DOWNLOAD_ROOT = Path("/content/aerial_batch_input")
DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_CACHE_ROOT = Path("/content/aerial_final_model_cache")
MODEL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)


def is_http_url(value) -> bool:
    return isinstance(value, str) and value.strip().lower().startswith(
        ("http://", "https://")
    )


def file_looks_valid(path: Path, minimum_bytes: int = 1024 * 1024) -> bool:
    return (
        path.exists()
        and path.is_file()
        and path.stat().st_size >= minimum_bytes
    )


def resolve_model_source(
    source,
    target_filename: str,
    minimum_bytes: int = 1024 * 1024,
) -> Path:
    """
    Resolve a model from either:
    - a mounted local/Google Drive path, or
    - a Google Drive shared file URL.
    """

    if source is None:
        raise ValueError(
            f"Model source is empty for {target_filename}."
        )

    source_text = str(source).strip()

    if not source_text:
        raise ValueError(
            f"Model source is empty for {target_filename}."
        )

    if is_http_url(source_text):
        target_path = MODEL_CACHE_ROOT / target_filename

        if file_looks_valid(
            target_path,
            minimum_bytes=minimum_bytes,
        ):
            print(
                f"{target_filename}: cached model found. "
                f"Skipping download."
            )
            return target_path

        print(
            f"Downloading model from Google Drive shared link:\n"
            f"{source_text}"
        )

        downloaded = gdown.download(
            url=source_text,
            output=str(target_path),
            quiet=False,
            fuzzy=True,
        )

        if downloaded is None:
            raise RuntimeError(
                f"gdown could not download model: {source_text}"
            )

        downloaded_path = Path(downloaded)

        if downloaded_path != target_path and downloaded_path.exists():
            shutil.move(
                str(downloaded_path),
                str(target_path),
            )

        if not file_looks_valid(
            target_path,
            minimum_bytes=minimum_bytes,
        ):
            size = (
                target_path.stat().st_size
                if target_path.exists()
                else 0
            )

            raise RuntimeError(
                f"Downloaded model file is missing or unexpectedly small: "
                f"{target_path} ({size} bytes). "
                "Check Google Drive sharing permissions."
            )

        print(
            f"Model ready: {target_path} "
            f"({target_path.stat().st_size / 1024 / 1024:.2f} MiB)"
        )

        return target_path

    local_path = Path(source_text)

    if not local_path.exists():
        raise FileNotFoundError(
            f"Local model path was not found: {local_path}"
        )

    if not local_path.is_file():
        raise FileNotFoundError(
            f"Model source is not a file: {local_path}"
        )

    print(
        f"Using mounted model: {local_path} "
        f"({local_path.stat().st_size / 1024 / 1024:.2f} MiB)"
    )

    return local_path


VALID_RUN_MODELS = {
    "YOLO26s",
    "RT-DETR-R18",
    "BPD-YOLOn/L-FPN",
    "ALL",
}

if RUN_MODEL not in VALID_RUN_MODELS:
    raise ValueError(
        f"RUN_MODEL must be one of: {sorted(VALID_RUN_MODELS)}"
    )

models_to_run = (
    [
        "YOLO26s",
        "RT-DETR-R18",
        "BPD-YOLOn/L-FPN",
    ]
    if RUN_MODEL == "ALL"
    else [RUN_MODEL]
)

MODEL_SOURCES = {
    "YOLO26s": YOLO26S_MODEL_SOURCE,
    "RT-DETR-R18": RTDETR_MODEL_SOURCE,
    "BPD-YOLOn/L-FPN": BPD_MODEL_SOURCE,
}

MODEL_FILENAMES = {
    "YOLO26s": "yolo26s_best.pt",
    "RT-DETR-R18": "rtdetr_r18_best.pth",
    "BPD-YOLOn/L-FPN": "bpd_yolon_lfpn_best.pt",
}

MODEL_PATHS = {}

for model_name in models_to_run:
    print("\n" + "=" * 88)
    print(f"RESOLVING MODEL: {model_name}")
    print("=" * 88)

    MODEL_PATHS[model_name] = resolve_model_source(
        source=MODEL_SOURCES[model_name],
        target_filename=MODEL_FILENAMES[model_name],
    )


YOLO26S_MODEL_PATH = MODEL_PATHS.get("YOLO26s")
RTDETR_MODEL_PATH = MODEL_PATHS.get("RT-DETR-R18")
BPD_MODEL_PATH = MODEL_PATHS.get("BPD-YOLOn/L-FPN")


def download_gdrive_shared_file(
    url: str,
    destination_dir: Path,
) -> Path:
    destination_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    output_path = destination_dir / "shared_input_file"

    path = gdown.download(
        url=url,
        output=str(output_path),
        quiet=False,
        fuzzy=True,
    )

    if path is None:
        raise RuntimeError(
            "gdown failed to download the shared media file."
        )

    path = Path(path)

    if path.is_dir():
        raise RuntimeError(
            "A directory was returned while a file was expected."
        )

    return path


def download_gdrive_shared_folder(
    url: str,
    destination_dir: Path,
) -> Path:
    destination_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    files = gdown.download_folder(
        url=url,
        output=str(destination_dir),
        quiet=False,
        remaining_ok=True,
        use_cookies=False,
    )

    if files is None:
        raise RuntimeError(
            "gdown failed to download the shared folder."
        )

    return destination_dir


def resolve_input_source():
    mode = INPUT_SOURCE_MODE.strip()

    if mode == "LOCAL_FOLDER":
        if not LOCAL_INPUT_PATH.exists():
            raise FileNotFoundError(
                f"Local input folder was not found: {LOCAL_INPUT_PATH}"
            )

        return LOCAL_INPUT_PATH

    if mode == "GDRIVE_SHARED_FOLDER":
        if not GDRIVE_SHARED_URL.strip():
            raise ValueError(
                "GDRIVE_SHARED_URL is empty."
            )

        target_dir = DOWNLOAD_ROOT / "shared_folder"

        if target_dir.exists():
            shutil.rmtree(
                target_dir
            )

        return download_gdrive_shared_folder(
            GDRIVE_SHARED_URL.strip(),
            target_dir,
        )

    if mode == "GDRIVE_SHARED_FILE":
        if not GDRIVE_SHARED_URL.strip():
            raise ValueError(
                "GDRIVE_SHARED_URL is empty."
            )

        target_dir = DOWNLOAD_ROOT / "shared_file"

        if target_dir.exists():
            shutil.rmtree(
                target_dir
            )

        target_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        return download_gdrive_shared_file(
            GDRIVE_SHARED_URL.strip(),
            target_dir,
        )

    raise ValueError(
        "INPUT_SOURCE_MODE must be one of: "
        "'LOCAL_FOLDER', 'GDRIVE_SHARED_FOLDER', "
        "'GDRIVE_SHARED_FILE'"
    )


INPUT_SOURCE_PATH = resolve_input_source()

print("\n" + "=" * 88)
print("MODEL + INPUT SOURCE RESOLUTION: PASSED")
print("=" * 88)

for model_name, model_path in MODEL_PATHS.items():
    print(
        f"{model_name:18s}: {model_path}"
    )

print("Input source       :", INPUT_SOURCE_PATH)
print("Exists             :", INPUT_SOURCE_PATH.exists())
print("Is file            :", INPUT_SOURCE_PATH.is_file())
print("Is dir             :", INPUT_SOURCE_PATH.is_dir())
print("=" * 88)


RESOLVING MODEL: YOLO26s
https://drive.google.com/file/d/1Ds2wuTMp1voGOH-VbNhqFfM-xwhpvBcL/view?usp=drive_link


Downloading...
From: https://drive.google.com/uc?id=1Ds2wuTMp1voGOH-VbNhqFfM-xwhpvBcL
To: /content/aerial_final_model_cache/yolo26s_best.pt
100%|██████████| 20.4M/20.4M [00:00<00:00, 163MB/s]


Model ready: /content/aerial_final_model_cache/yolo26s_best.pt (19.43 MiB)

RESOLVING MODEL: RT-DETR-R18
https://drive.google.com/file/d/1Ku_Sh9i-aCN2aLaDQH8tDfgrTzBFzZ3M/view?usp=drive_link


Downloading...
From (original): https://drive.google.com/uc?id=1Ku_Sh9i-aCN2aLaDQH8tDfgrTzBFzZ3M
From (redirected): https://drive.google.com/uc?id=1Ku_Sh9i-aCN2aLaDQH8tDfgrTzBFzZ3M&confirm=t&uuid=0d45fb75-5c06-4187-9bfa-3cc874644d50
To: /content/aerial_final_model_cache/rtdetr_r18_best.pth
100%|██████████| 322M/322M [00:01<00:00, 181MB/s]


Model ready: /content/aerial_final_model_cache/rtdetr_r18_best.pth (307.19 MiB)

RESOLVING MODEL: BPD-YOLOn/L-FPN
https://drive.google.com/file/d/1sV5nb8uU4E_b_7JDitpbYbxoM0x2ob2H/view?usp=drive_link


Downloading...
From: https://drive.google.com/uc?id=1sV5nb8uU4E_b_7JDitpbYbxoM0x2ob2H
To: /content/aerial_final_model_cache/bpd_yolon_lfpn_best.pt
100%|██████████| 3.81M/3.81M [00:00<00:00, 315MB/s]

Model ready: /content/aerial_final_model_cache/bpd_yolon_lfpn_best.pt (3.64 MiB)

MODEL + INPUT SOURCE RESOLUTION: PASSED
YOLO26s           : /content/aerial_final_model_cache/yolo26s_best.pt
RT-DETR-R18       : /content/aerial_final_model_cache/rtdetr_r18_best.pth
BPD-YOLOn/L-FPN   : /content/aerial_final_model_cache/bpd_yolon_lfpn_best.pt
Input source       : /content/drive/MyDrive/referee_videos
Exists             : True
Is file            : False
Is dir             : True


## 6. Collect all media files from the input source

In [6]:
def collect_media_files(source_path: Path):
    if source_path.is_file():
        if not is_supported_media(source_path):
            raise ValueError(f"The file is not a supported image/video: {source_path}")
        return [source_path]

    if not source_path.is_dir():
        raise ValueError(f"Input source is neither file nor directory: {source_path}")

    candidates = source_path.rglob("*") if RECURSIVE_SCAN else source_path.glob("*")
    media_files = [p for p in candidates if is_supported_media(p)]
    media_files = sorted(media_files)

    if LIMIT_MEDIA_COUNT is not None:
        media_files = media_files[: int(LIMIT_MEDIA_COUNT)]

    if not media_files:
        raise RuntimeError(f"No supported image/video files were found inside: {source_path}")

    return media_files


MEDIA_FILES = collect_media_files(INPUT_SOURCE_PATH)

print(f"Collected {len(MEDIA_FILES)} supported media file(s).\\n")
for idx, p in enumerate(MEDIA_FILES, start=1):
    print(f"{idx:03d} | {p}")

Collected 11 supported media file(s).\n
001 | /content/drive/MyDrive/referee_videos/1.mp4
002 | /content/drive/MyDrive/referee_videos/v4 - frame at 0m12s.jpg
003 | /content/drive/MyDrive/referee_videos/v4 - frame at 0m1s.jpg
004 | /content/drive/MyDrive/referee_videos/v4 - frame at 0m25s.jpg
005 | /content/drive/MyDrive/referee_videos/vid3 - frame at 0m0s.jpg
006 | /content/drive/MyDrive/referee_videos/vid3 - frame at 0m10s.jpg
007 | /content/drive/MyDrive/referee_videos/vid3 - frame at 0m14s.jpg
008 | /content/drive/MyDrive/referee_videos/vid3 - frame at 0m20s.jpg
009 | /content/drive/MyDrive/referee_videos/vid3 - frame at 0m21s.jpg
010 | /content/drive/MyDrive/referee_videos/vid3 - frame at 0m23s.jpg
011 | /content/drive/MyDrive/referee_videos/vid3 - frame at 0m7s.jpg


## 7. BPD compatibility (`DySample`)

In [7]:
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F


class DySample(nn.Module):
    def __init__(self, channels=None, scale=2, max_offset=0.25):
        super().__init__()
        self.channels = int(channels) if channels is not None else None
        self.scale = int(scale)
        self.max_offset = float(max_offset)

        if self.channels is None:
            self.offset = nn.LazyConv2d(2, kernel_size=1, stride=1, padding=0)
            self._zero_initialized = False
        else:
            self.offset = nn.Conv2d(self.channels, 2, kernel_size=1, stride=1, padding=0)
            with torch.no_grad():
                nn.init.zeros_(self.offset.weight)
                if self.offset.bias is not None:
                    nn.init.zeros_(self.offset.bias)

    def _materialize_legacy_lazy_offset(self, x):
        if isinstance(self.offset, nn.LazyConv2d) and not getattr(self, "_zero_initialized", False):
            _ = self.offset(x)
            with torch.no_grad():
                nn.init.zeros_(self.offset.weight)
                if self.offset.bias is not None:
                    nn.init.zeros_(self.offset.bias)
            self._zero_initialized = True

    def forward(self, x):
        channels = getattr(self, "channels", None)
        if channels is not None and x.shape[1] != int(channels):
            raise RuntimeError(f"DySample expected {int(channels)} channels but received {x.shape[1]}.")

        self._materialize_legacy_lazy_offset(x)

        b, _, h, w = x.shape
        scale = int(getattr(self, "scale", 2))
        max_offset = float(getattr(self, "max_offset", 0.25))
        oh = h * scale
        ow = w * scale

        raw = self.offset(x)
        raw = F.interpolate(raw, size=(oh, ow), mode="bilinear", align_corners=False)
        raw = torch.tanh(raw) * max_offset

        ys = ((torch.arange(oh, device=x.device, dtype=x.dtype) + 0.5) / oh) * 2.0 - 1.0
        xs = ((torch.arange(ow, device=x.device, dtype=x.dtype) + 0.5) / ow) * 2.0 - 1.0

        yy, xx = torch.meshgrid(ys, xs, indexing="ij")
        base = torch.stack((xx, yy), dim=-1).unsqueeze(0).expand(b, -1, -1, -1)

        dx = raw[:, 0] * (2.0 / max(w, 1))
        dy = raw[:, 1] * (2.0 / max(h, 1))
        grid = base + torch.stack((dx, dy), dim=-1)

        return F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)


def register_bpd_compatibility():
    import ultralytics.nn.tasks as ultralytics_tasks

    setattr(sys.modules["__main__"], "DySample", DySample)
    setattr(ultralytics_tasks, "DySample", DySample)

    try:
        import ultralytics.nn.modules as ultralytics_modules
        setattr(ultralytics_modules, "DySample", DySample)
    except Exception:
        pass

    try:
        torch.serialization.add_safe_globals([DySample])
    except Exception:
        pass


register_bpd_compatibility()
print("BPD DySample compatibility: REGISTERED")

BPD DySample compatibility: REGISTERED


## 8. Common ByteTrack wrapper

In [8]:
from ultralytics.engine.results import Boxes
from ultralytics.trackers.byte_tracker import BYTETracker


class CommonByteTracker:
    def __init__(self, frame_rate=30.0):
        args = SimpleNamespace(
            tracker_type="bytetrack",
            track_high_thresh=TRACK_HIGH_THRESH,
            track_low_thresh=TRACK_LOW_THRESH,
            new_track_thresh=NEW_TRACK_THRESH,
            track_buffer=TRACK_BUFFER,
            match_thresh=MATCH_THRESH,
            fuse_score=FUSE_SCORE,
        )

        try:
            self.tracker = BYTETracker(args, frame_rate=frame_rate)
        except TypeError:
            self.tracker = BYTETracker(args)

    def update(self, detections: DetectionBatch, frame):
        height, width = frame.shape[:2]

        if len(detections) == 0:
            data = torch.empty((0, 6), dtype=torch.float32)
        else:
            class_column = np.zeros((len(detections), 1), dtype=np.float32)
            data_np = np.concatenate(
                [
                    detections.boxes.astype(np.float32),
                    detections.scores.reshape(-1, 1).astype(np.float32),
                    class_column,
                ],
                axis=1,
            )
            data = torch.from_numpy(data_np)

        boxes = Boxes(data, (height, width))
        tracks = self.tracker.update(boxes, frame)

        if tracks is None or len(tracks) == 0:
            return (
                np.zeros((0, 4), dtype=np.float32),
                np.zeros((0,), dtype=np.int32),
                np.zeros((0,), dtype=np.float32),
            )

        tracks = np.asarray(tracks, dtype=np.float32)
        if tracks.ndim == 1:
            tracks = tracks[None, :]

        track_boxes = tracks[:, :4]
        track_ids = tracks[:, 4].astype(np.int32)
        track_scores = tracks[:, 5]

        return track_boxes, track_ids, track_scores


print("Common ByteTrack wrapper: READY")

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 181ms
Prepared 1 package in 36ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Common ByteTrack wrapper: READY


## 9. Detector backends

In [9]:
# ============================================================
# DETECTOR BACKENDS
# YOLO26s / RT-DETR-R18 / BPD-YOLOn-LFPN
# ============================================================

from pathlib import Path

import contextlib
import gc
import importlib
import os
import subprocess
import sys
import time

import cv2
import numpy as np
import torch

from PIL import Image
from ultralytics import YOLO


# ============================================================
# HELPER: INSTALL RT-DETR RUNTIME DEPENDENCY IF NEEDED
# ============================================================

def ensure_faster_coco_eval():
    try:
        import faster_coco_eval
        print("faster-coco-eval: READY")
        return

    except ImportError:
        print("Installing faster-coco-eval...")

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "faster-coco-eval>=1.6.7",
            ]
        )

        importlib.invalidate_caches()

        import faster_coco_eval

        print("faster-coco-eval: INSTALLED")


# ============================================================
# HELPER: SAFELY SET SINGLE CLASS NAME
# ============================================================

def set_person_class_name(yolo_model):
    """
    Set the single class name on the underlying Ultralytics model.
    YOLO.names itself is a read-only property.
    """

    try:
        if (
            hasattr(yolo_model, "model")
            and hasattr(yolo_model.model, "names")
        ):
            yolo_model.model.names = {
                0: "person"
            }

    except Exception as exc:
        print(
            f"Class-name override skipped: {exc}"
        )


# ============================================================
# YOLO / BPD BACKEND
# ============================================================

class UltralyticsDetector:
    def __init__(
        self,
        model_path: Path,
        model_name: str,
        is_bpd: bool = False,
    ):
        self.model_path = Path(
            model_path
        )

        self.model_name = model_name
        self.is_bpd = is_bpd

        if not self.model_path.exists():
            raise FileNotFoundError(
                f"Model checkpoint was not found: {self.model_path}"
            )

        if is_bpd:
            register_bpd_compatibility()

        print(
            f"Loading {model_name} checkpoint:"
        )
        print(
            self.model_path
        )

        self.model = YOLO(
            str(
                self.model_path
            )
        )

        set_person_class_name(
            self.model
        )

        print(
            f"{model_name} checkpoint loaded successfully."
        )

    def predict(
        self,
        frame,
    ) -> DetectionBatch:

        if frame is None:
            raise ValueError(
                "Input frame is None."
            )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.perf_counter()

        # Do not pass the deprecated 'half' argument.
        # The reference PyTorch inference is kept in standard precision.
        results = self.model.predict(
            source=frame,
            imgsz=IMAGE_SIZE,
            conf=CONF_THRESHOLD,
            iou=IOU_THRESHOLD,
            max_det=MAX_DETECTIONS,
            classes=[0],
            device=DEVICE,
            verbose=False,
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed_ms = (
            time.perf_counter()
            - start
        ) * 1000.0

        if not results:
            return DetectionBatch.empty(
                elapsed_ms
            )

        result = results[0]

        if (
            result.boxes is None
            or len(
                result.boxes
            ) == 0
        ):
            return DetectionBatch.empty(
                elapsed_ms
            )

        boxes = (
            result.boxes.xyxy
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32
            )
        )

        scores = (
            result.boxes.conf
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32
            )
        )

        classes = np.zeros(
            (
                len(
                    boxes
                ),
            ),
            dtype=np.float32,
        )

        return DetectionBatch(
            boxes=boxes,
            scores=scores,
            classes=classes,
            detector_ms=elapsed_ms,
        )


# ============================================================
# RT-DETR-R18 BACKEND
# ============================================================

class RTDETRR18Detector:
    def __init__(
        self,
        model_path: Path,
    ):
        self.model_path = Path(
            model_path
        )

        self.model_name = (
            "RT-DETR-R18"
        )

        if not self.model_path.exists():
            raise FileNotFoundError(
                f"RT-DETR checkpoint was not found: {self.model_path}"
            )

        self.local_root = Path(
            "/content/aerial_rtdetr_inference"
        )

        self.repo_root = (
            self.local_root
            / "RT-DETR"
        )

        self.rtdetr_root = (
            self.repo_root
            / "rtdetrv2_pytorch"
        )

        ensure_faster_coco_eval()

        self._prepare_repository()
        self._build_model()

    # --------------------------------------------------------
    # PREPARE OFFICIAL RT-DETR REPOSITORY
    # --------------------------------------------------------

    def _prepare_repository(
        self,
    ):
        self.local_root.mkdir(
            parents=True,
            exist_ok=True,
        )

        if not self.repo_root.exists():

            print(
                "Cloning official RT-DETR repository..."
            )

            subprocess.check_call(
                [
                    "git",
                    "clone",
                    "--filter=blob:none",
                    RTDETR_REPO_URL,
                    str(
                        self.repo_root
                    ),
                ]
            )

        print(
            "Preparing pinned RT-DETR source..."
        )

        subprocess.check_call(
            [
                "git",
                "-C",
                str(
                    self.repo_root
                ),
                "fetch",
                "--all",
                "--tags",
                "--prune",
            ]
        )

        subprocess.check_call(
            [
                "git",
                "-C",
                str(
                    self.repo_root
                ),
                "checkout",
                RTDETR_REPO_COMMIT,
            ]
        )

        if not self.rtdetr_root.exists():
            raise FileNotFoundError(
                f"RT-DETR Python root was not found: {self.rtdetr_root}"
            )

        print(
            "RT-DETR repository: READY"
        )

    # --------------------------------------------------------
    # EXTRACT CHECKPOINT STATE DICTIONARY
    # --------------------------------------------------------

    @staticmethod
    def _extract_state_dict(
        checkpoint,
    ):
        if not isinstance(
            checkpoint,
            dict,
        ):
            raise RuntimeError(
                "Unexpected RT-DETR checkpoint format."
            )

        # Prefer EMA weights.
        ema = checkpoint.get(
            "ema"
        )

        if ema is not None:

            if isinstance(
                ema,
                dict,
            ):

                if "module" in ema:
                    module_state = ema[
                        "module"
                    ]

                    if isinstance(
                        module_state,
                        dict,
                    ):
                        return module_state

                # Some checkpoints store the state dictionary
                # directly under the 'ema' key.
                if all(
                    isinstance(
                        key,
                        str,
                    )
                    for key
                    in ema.keys()
                ):
                    return ema

            if hasattr(
                ema,
                "state_dict",
            ):
                return ema.state_dict()

        # Fallback to regular model weights.
        model_state = checkpoint.get(
            "model"
        )

        if model_state is not None:

            if isinstance(
                model_state,
                dict,
            ):
                return model_state

            if hasattr(
                model_state,
                "state_dict",
            ):
                return model_state.state_dict()

        raise RuntimeError(
            "Could not find RT-DETR model weights "
            "under 'ema' or 'model' in the checkpoint."
        )

    # --------------------------------------------------------
    # BUILD MODEL
    # --------------------------------------------------------

    def _build_model(
        self,
    ):
        if (
            str(
                self.rtdetr_root
            )
            not in sys.path
        ):
            sys.path.insert(
                0,
                str(
                    self.rtdetr_root
                ),
            )

        # Remove previously cached RT-DETR src modules if needed.
        importlib.invalidate_caches()

        from src.core import YAMLConfig

        config_dir = (
            self.rtdetr_root
            / "configs"
            / "custom"
        )

        config_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        config_path = (
            config_dir
            / "rtdetr_r18_visdrone_person_inference.yml"
        )

        # IMPORTANT:
        # The final newline below must be a real "\n".
        # Do not replace it with "\\n".
        config_text = f"""
__include__:
  - ../rtdetr/rtdetr_r18vd_6x_coco.yml

num_classes: 1
remap_mscoco_category: False

eval_spatial_size: [{IMAGE_SIZE}, {IMAGE_SIZE}]

PResNet:
  pretrained: False
"""

        config_path.write_text(
            config_text.strip()
            + "\n",
            encoding="utf-8",
        )

        # Validate generated YAML text before parsing.
        written_config = (
            config_path.read_text(
                encoding="utf-8"
            )
        )

        if "\\n" in written_config:
            raise RuntimeError(
                "Invalid literal backslash-n was found "
                "inside the generated RT-DETR YAML."
            )

        if (
            "pretrained: False"
            not in written_config
        ):
            raise RuntimeError(
                "RT-DETR YAML validation failed: "
                "'pretrained: False' was not found."
            )

        print(
            "\nRT-DETR inference configuration:"
        )
        print(
            "-" * 60
        )
        print(
            written_config
        )
        print(
            "-" * 60
        )

        old_cwd = Path.cwd()

        try:
            os.chdir(
                self.rtdetr_root
            )

            cfg = YAMLConfig(
                str(
                    config_path
                ),
                device=DEVICE,
                use_amp=(
                    USE_FP16
                    and torch.cuda.is_available()
                ),
            )

        finally:
            os.chdir(
                old_cwd
            )

        print(
            "RT-DETR architecture created."
        )

        print(
            "Loading RT-DETR checkpoint:"
        )
        print(
            self.model_path
        )

        checkpoint = torch.load(
            self.model_path,
            map_location="cpu",
            weights_only=False,
        )

        state = self._extract_state_dict(
            checkpoint
        )

        load_result = (
            cfg.model.load_state_dict(
                state,
                strict=True,
            )
        )

        print(
            "RT-DETR state dictionary loaded."
        )

        self.model = (
            cfg.model
            .deploy()
            .to(
                DEVICE
            )
            .eval()
        )

        self.postprocessor = (
            cfg.postprocessor
            .deploy()
        )

        del checkpoint
        del state
        del cfg

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

        print(
            "RT-DETR-R18 checkpoint loaded successfully."
        )

    # --------------------------------------------------------
    # PREPROCESS
    # --------------------------------------------------------

    def _preprocess(
        self,
        frame,
    ):
        if frame is None:
            raise ValueError(
                "Input frame is None."
            )

        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB,
        )

        pil = Image.fromarray(
            rgb
        )

        pil = pil.resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE,
            ),
            Image.Resampling.BILINEAR,
        )

        array = (
            np.asarray(
                pil,
                dtype=np.float32,
            )
            / 255.0
        )

        tensor = (
            torch.from_numpy(
                array
            )
            .permute(
                2,
                0,
                1,
            )
            .unsqueeze(
                0
            )
            .contiguous()
        )

        return tensor.to(
            DEVICE,
            non_blocking=True,
        )

    # --------------------------------------------------------
    # INFERENCE
    # --------------------------------------------------------

    def predict(
        self,
        frame,
    ) -> DetectionBatch:

        if frame is None:
            raise ValueError(
                "Input frame is None."
            )

        height, width = (
            frame.shape[:2]
        )

        tensor = self._preprocess(
            frame
        )

        # RT-DETR expects original target size as [width, height].
        original_size = torch.tensor(
            [
                [
                    width,
                    height,
                ]
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.perf_counter()

        with torch.inference_mode():

            if (
                USE_FP16
                and torch.cuda.is_available()
            ):
                autocast_context = (
                    torch.autocast(
                        device_type="cuda",
                        dtype=torch.float16,
                    )
                )

            else:
                autocast_context = (
                    contextlib.nullcontext()
                )

            with autocast_context:

                outputs = self.model(
                    tensor
                )

                (
                    labels,
                    boxes,
                    scores,
                ) = self.postprocessor(
                    outputs,
                    original_size,
                )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed_ms = (
            time.perf_counter()
            - start
        ) * 1000.0

        labels = (
            labels[0]
            .detach()
            .cpu()
            .numpy()
        )

        boxes = (
            boxes[0]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32
            )
        )

        scores = (
            scores[0]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32
            )
        )

        # The trained RT-DETR model contains one class:
        # class 0 = person.
        mask = (
            (scores >= CONF_THRESHOLD)
            & (labels == 0)
        )

        boxes = boxes[
            mask
        ]

        scores = scores[
            mask
        ]

        if len(
            scores
        ) > MAX_DETECTIONS:

            order = np.argsort(
                -scores
            )[
                :MAX_DETECTIONS
            ]

            boxes = boxes[
                order
            ]

            scores = scores[
                order
            ]

        if len(
            boxes
        ) == 0:
            return DetectionBatch.empty(
                elapsed_ms
            )

        classes = np.zeros(
            (
                len(
                    boxes
                ),
            ),
            dtype=np.float32,
        )

        return DetectionBatch(
            boxes=boxes,
            scores=scores,
            classes=classes,
            detector_ms=elapsed_ms,
        )


# ============================================================
# UNIFIED DETECTOR BUILDER
# ============================================================

def build_detector(
    model_name: str,
):

    if model_name == "YOLO26s":

        if YOLO26S_MODEL_PATH is None:
            raise RuntimeError(
                "YOLO26S_MODEL_PATH has not been resolved."
            )

        return UltralyticsDetector(
            model_path=YOLO26S_MODEL_PATH,
            model_name="YOLO26s",
            is_bpd=False,
        )

    if model_name == "RT-DETR-R18":

        if RTDETR_MODEL_PATH is None:
            raise RuntimeError(
                "RTDETR_MODEL_PATH has not been resolved."
            )

        return RTDETRR18Detector(
            model_path=RTDETR_MODEL_PATH,
        )

    if model_name == "BPD-YOLOn/L-FPN":

        if BPD_MODEL_PATH is None:
            raise RuntimeError(
                "BPD_MODEL_PATH has not been resolved."
            )

        return UltralyticsDetector(
            model_path=BPD_MODEL_PATH,
            model_name="BPD-YOLOn/L-FPN",
            is_bpd=True,
        )

    raise ValueError(
        f"Unsupported model: {model_name}"
    )


# ============================================================
# READY
# ============================================================

print(
    "=" * 72
)

print(
    "Detector backends: READY"
)

print(
    "Supported models:"
)

print(
    " - YOLO26s"
)

print(
    " - RT-DETR-R18"
)

print(
    " - BPD-YOLOn/L-FPN"
)

print(
    "=" * 72
)

Detector backends: READY
Supported models:
 - YOLO26s
 - RT-DETR-R18
 - BPD-YOLOn/L-FPN


## 10. Preflight

In [10]:
if not MEDIA_FILES:
    raise RuntimeError(
        "MEDIA_FILES is empty."
    )

for model_name in models_to_run:
    model_path = MODEL_PATHS.get(
        model_name
    )

    if model_path is None:
        raise RuntimeError(
            f"Resolved checkpoint is missing for {model_name}."
        )

    if not model_path.exists():
        raise FileNotFoundError(
            f"{model_name} checkpoint was not found after resolution: "
            f"{model_path}"
        )

    print(
        f"{model_name:18s} -> "
        f"{model_path} "
        f"({model_path.stat().st_size / 1024 / 1024:.2f} MiB)"
    )


print("\nInput source :", INPUT_SOURCE_PATH)
print("Media count   :", len(MEDIA_FILES))
print("Models        :", models_to_run)
print("Output root   :", OUTPUT_ROOT)
print("\nPRE-FLIGHT: PASSED")

YOLO26s            -> /content/aerial_final_model_cache/yolo26s_best.pt (19.43 MiB)
RT-DETR-R18        -> /content/aerial_final_model_cache/rtdetr_r18_best.pth (307.19 MiB)
BPD-YOLOn/L-FPN    -> /content/aerial_final_model_cache/bpd_yolon_lfpn_best.pt (3.64 MiB)

Input source : /content/drive/MyDrive/referee_videos
Media count   : 11
Models        : ['YOLO26s', 'RT-DETR-R18', 'BPD-YOLOn/L-FPN']
Output root   : /content/drive/MyDrive/aerial_human_detection/batch_inference_outputs_L4_V2

PRE-FLIGHT: PASSED


## 11. Warmup helper

In [11]:
def warmup_detector(detector, frame):
    if WARMUP_RUNS <= 0:
        return

    print(f"Warming up {detector.model_name} for {WARMUP_RUNS} run(s)...")

    for _ in range(WARMUP_RUNS):
        _ = detector.predict(frame)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print("Warm-up: COMPLETE")

## 12. Single image processing

In [12]:
def process_image(model_name: str, detector, media_path: Path, run_dir: Path):
    image = cv2.imread(str(media_path))
    if image is None:
        raise RuntimeError(f"OpenCV could not read image: {media_path}")

    warmup_detector(detector, image)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    detections = detector.predict(image)

    mean_conf = float(detections.scores.mean()) if len(detections) else 0.0

    annotated = draw_detection_boxes(image, detections.boxes, detections.scores)

    canvas = build_image_footer(
        annotated=annotated,
        model_name=model_name,
        model_path=MODEL_PATHS[model_name],
        detector_ms=detections.detector_ms,
        detections=len(detections),
        mean_conf=mean_conf,
    )

    relative_stub = sanitize_name(media_path.stem)
    image_output = run_dir / "images" / f"{relative_stub}__detected.jpg"
    csv_output = run_dir / "csv" / f"{relative_stub}__detections.csv"
    json_output = run_dir / "json" / f"{relative_stub}__summary.json"

    image_output.parent.mkdir(parents=True, exist_ok=True)
    csv_output.parent.mkdir(parents=True, exist_ok=True)
    json_output.parent.mkdir(parents=True, exist_ok=True)

    cv2.imwrite(str(image_output), canvas, [int(cv2.IMWRITE_JPEG_QUALITY), 95])

    rows = []
    for index, (box, score) in enumerate(zip(detections.boxes, detections.scores), start=1):
        x1, y1, x2, y2 = [float(v) for v in box]
        rows.append(
            {
                "detection_id": index,
                "class": "person",
                "confidence": float(score),
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "width": x2 - x1,
                "height": y2 - y1,
            }
        )

    if SAVE_CSV:
        pd.DataFrame(rows).to_csv(csv_output, index=False)

    max_gpu_mb = (
        torch.cuda.max_memory_allocated() / 1024 / 1024
        if torch.cuda.is_available()
        else 0.0
    )

    summary = {
        "media_type": "image",
        "source": str(media_path),
        "model": model_name,
        "checkpoint": str(MODEL_PATHS[model_name]),
        "checkpoint_sha256": sha256_file(MODEL_PATHS[model_name]),
        "device": DEVICE,
        "fp16": USE_FP16,
        "imgsz": IMAGE_SIZE,
        "conf_threshold": CONF_THRESHOLD,
        "iou_threshold": IOU_THRESHOLD,
        "max_detections": MAX_DETECTIONS,
        "detections": len(detections),
        "mean_detection_confidence": mean_conf,
        "detector_latency_ms": detections.detector_ms,
        "detector_throughput_fps": (1000.0 / detections.detector_ms if detections.detector_ms > 0 else 0.0),
        "max_torch_gpu_memory_mb": max_gpu_mb,
        "step3_validation": STEP3_VALIDATION[model_name],
        "note": (
            "Detection confidence and throughput are measured on this media. "
            "Step3 validation metrics are reference VisDrone metrics, not "
            "ground-truth accuracy for this image."
        ),
        "output_image": str(image_output),
        "output_csv": str(csv_output) if SAVE_CSV else None,
    }

    if SAVE_JSON:
        json_output.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    return summary

## 13. Single video processing

In [13]:
def process_video(model_name: str, detector, media_path: Path, run_dir: Path):
    cap = cv2.VideoCapture(str(media_path))
    if not cap.isOpened():
        raise RuntimeError(f"OpenCV could not open video: {media_path}")

    source_fps = float(cap.get(cv2.CAP_PROP_FPS))
    if not np.isfinite(source_fps) or source_fps <= 0:
        source_fps = 25.0

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    first_ok, first_frame = cap.read()
    if not first_ok:
        cap.release()
        raise RuntimeError("The video contains no readable frames.")

    warmup_detector(detector, first_frame)
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    tracker = CommonByteTracker(frame_rate=source_fps)

    relative_stub = sanitize_name(media_path.stem)
    raw_output = run_dir / "videos" / f"{relative_stub}__ByteTrack_raw.mp4"
    csv_output = run_dir / "csv" / f"{relative_stub}__tracks.csv"
    json_output = run_dir / "json" / f"{relative_stub}__summary.json"

    raw_output.parent.mkdir(parents=True, exist_ok=True)
    csv_output.parent.mkdir(parents=True, exist_ok=True)
    json_output.parent.mkdir(parents=True, exist_ok=True)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(raw_output), fourcc, source_fps, (width, height))

    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f"Could not create output video: {raw_output}")

    trails = defaultdict(lambda: deque(maxlen=TRAIL_LENGTH))
    unique_track_ids = set()
    csv_rows = []
    processing_fps_history = deque(maxlen=FPS_SMOOTHING_WINDOW)
    detector_latency_history = []
    total_latency_history = []
    detection_confidences = []
    processed_frames = 0
    skipped_frames = 0

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    overall_start = time.perf_counter()
    progress_total = total_frames if total_frames > 0 else None
    progress = tqdm(total=progress_total, desc=f"{model_name} video", unit="frame")

    frame_index = 0

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            frame_index += 1

            if VIDEO_FRAME_STRIDE > 1 and (frame_index - 1) % VIDEO_FRAME_STRIDE != 0:
                skipped_frames += 1
                progress.update(1)
                continue

            frame_start = time.perf_counter()

            detections = detector.predict(frame)
            track_boxes, track_ids, track_scores = tracker.update(detections, frame)

            for box, track_id, score in zip(track_boxes, track_ids, track_scores):
                x1, y1, x2, y2 = [float(v) for v in box]
                center = (
                    int(round((x1 + x2) / 2)),
                    int(round((y1 + y2) / 2)),
                )
                trails[int(track_id)].append(center)
                unique_track_ids.add(int(track_id))

                csv_rows.append(
                    {
                        "frame": frame_index,
                        "time_seconds": ((frame_index - 1) / source_fps),
                        "track_id": int(track_id),
                        "class": "person",
                        "confidence": float(score),
                        "x1": x1,
                        "y1": y1,
                        "x2": x2,
                        "y2": y2,
                        "width": x2 - x1,
                        "height": y2 - y1,
                    }
                )

            if len(detections):
                detection_confidences.extend(detections.scores.tolist())

            annotated = draw_detection_boxes(
                frame=frame,
                boxes=track_boxes,
                scores=track_scores,
                track_ids=track_ids,
                trails=trails,
            )

            elapsed_before_overlay = (time.perf_counter() - frame_start)
            frame_processing_fps = (1.0 / elapsed_before_overlay if elapsed_before_overlay > 0 else 0.0)
            processing_fps_history.append(frame_processing_fps)
            smooth_fps = float(np.mean(processing_fps_history))
            total_ms = elapsed_before_overlay * 1000.0

            mean_frame_conf = float(detections.scores.mean()) if len(detections) else 0.0

            annotated = add_video_status_overlay(
                frame=annotated,
                model_name=model_name,
                frame_number=frame_index,
                source_fps=source_fps,
                processing_fps=smooth_fps,
                detector_ms=detections.detector_ms,
                total_ms=total_ms,
                detections=len(detections),
                active_tracks=len(track_ids),
                unique_tracks=len(unique_track_ids),
                mean_conf=mean_frame_conf,
            )

            writer.write(annotated)

            detector_latency_history.append(detections.detector_ms)
            total_latency_history.append(total_ms)
            processed_frames += 1

            progress.update(1)
            progress.set_postfix(
                {
                    "FPS": f"{smooth_fps:.1f}",
                    "IDs": len(unique_track_ids),
                    "det": len(detections),
                },
                refresh=False,
            )

    finally:
        progress.close()
        cap.release()
        writer.release()

    overall_seconds = time.perf_counter() - overall_start

    if SAVE_CSV:
        pd.DataFrame(csv_rows).to_csv(csv_output, index=False)

    final_video = maybe_transcode_h264(raw_output) if SAVE_VIDEO else raw_output

    if final_video != raw_output and raw_output.exists():
        try:
            raw_output.unlink()
        except Exception:
            pass

    avg_detector_ms = float(np.mean(detector_latency_history)) if detector_latency_history else 0.0
    median_detector_ms = float(np.median(detector_latency_history)) if detector_latency_history else 0.0
    avg_total_ms = float(np.mean(total_latency_history)) if total_latency_history else 0.0
    measured_processing_fps = processed_frames / overall_seconds if overall_seconds > 0 else 0.0
    mean_conf = float(np.mean(detection_confidences)) if detection_confidences else 0.0

    max_gpu_mb = (
        torch.cuda.max_memory_allocated() / 1024 / 1024
        if torch.cuda.is_available()
        else 0.0
    )

    source_duration = total_frames / source_fps if total_frames > 0 else None

    summary = {
        "media_type": "video",
        "source": str(media_path),
        "model": model_name,
        "checkpoint": str(MODEL_PATHS[model_name]),
        "checkpoint_sha256": sha256_file(MODEL_PATHS[model_name]),
        "device": DEVICE,
        "fp16": USE_FP16,
        "imgsz": IMAGE_SIZE,
        "conf_threshold": CONF_THRESHOLD,
        "iou_threshold": IOU_THRESHOLD,
        "max_detections": MAX_DETECTIONS,
        "bytetrack": {
            "track_high_thresh": TRACK_HIGH_THRESH,
            "track_low_thresh": TRACK_LOW_THRESH,
            "new_track_thresh": NEW_TRACK_THRESH,
            "track_buffer": TRACK_BUFFER,
            "match_thresh": MATCH_THRESH,
            "fuse_score": FUSE_SCORE,
            "trail_length": TRAIL_LENGTH,
        },
        "source_video": {
            "fps": source_fps,
            "frames": total_frames,
            "width": width,
            "height": height,
            "duration_seconds": source_duration,
        },
        "runtime": {
            "processed_frames": processed_frames,
            "skipped_frames": skipped_frames,
            "overall_processing_seconds": overall_seconds,
            "effective_processing_fps": measured_processing_fps,
            "average_detector_latency_ms": avg_detector_ms,
            "median_detector_latency_ms": median_detector_ms,
            "average_total_frame_latency_ms": avg_total_ms,
            "max_torch_gpu_memory_mb": max_gpu_mb,
        },
        "detections_and_tracking": {
            "track_rows": len(csv_rows),
            "unique_track_ids": len(unique_track_ids),
            "mean_detection_confidence": mean_conf,
        },
        "step3_validation": STEP3_VALIDATION[model_name],
        "note": (
            "Runtime FPS/latency/confidence are measured on this video. "
            "Step3 validation mAP/AP50 are reference VisDrone validation metrics. "
            "No ground-truth tracking accuracy metric such as HOTA/IDF1/MOTA "
            "can be computed without annotated tracking ground truth."
        ),
        "output_video": str(final_video),
        "output_csv": str(csv_output) if SAVE_CSV else None,
    }

    if SAVE_JSON:
        json_output.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    return summary

In [14]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "faster-coco-eval>=1.6.7",
    ]
)

import faster_coco_eval

print("faster-coco-eval installed successfully.")

faster-coco-eval installed successfully.


## 14. Global batch processing over the whole folder

In [15]:
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
SESSION_ROOT = OUTPUT_ROOT / f"batch_run_{RUN_TIMESTAMP}"
SESSION_ROOT.mkdir(parents=True, exist_ok=True)

all_summaries = []
comparison_rows = []

for model_name in models_to_run:
    print("\\n" + "#" * 100)
    print(f"LOADING MODEL: {model_name}")
    print("#" * 100)

    detector = build_detector(model_name)
    print(f"{model_name} loaded successfully.")

    model_slug = sanitize_name(model_name)
    model_run_dir = SESSION_ROOT / model_slug
    model_run_dir.mkdir(parents=True, exist_ok=True)

    media_progress = tqdm(MEDIA_FILES, desc=f"{model_name} batch", unit="file")

    try:
        for media_path in media_progress:
            media_type = classify_media(media_path)
            media_progress.set_postfix({"type": media_type}, refresh=False)

            try:
                if media_type == "image":
                    summary = process_image(
                        model_name=model_name,
                        detector=detector,
                        media_path=media_path,
                        run_dir=model_run_dir,
                    )
                else:
                    summary = process_video(
                        model_name=model_name,
                        detector=detector,
                        media_path=media_path,
                        run_dir=model_run_dir,
                    )

                all_summaries.append(summary)

                row = {
                    "model": summary["model"],
                    "source": summary["source"],
                    "media_type": summary["media_type"],
                    "step3_val_map50_95": summary["step3_validation"]["map50_95"],
                    "step3_val_ap50": summary["step3_validation"]["ap50"],
                }

                if summary["media_type"] == "video":
                    row.update(
                        {
                            "processing_fps": summary["runtime"]["effective_processing_fps"],
                            "detector_latency_ms": summary["runtime"]["average_detector_latency_ms"],
                            "total_frame_latency_ms": summary["runtime"]["average_total_frame_latency_ms"],
                            "unique_track_ids": summary["detections_and_tracking"]["unique_track_ids"],
                            "mean_detection_confidence": summary["detections_and_tracking"]["mean_detection_confidence"],
                        }
                    )
                else:
                    row.update(
                        {
                            "processing_fps": summary["detector_throughput_fps"],
                            "detector_latency_ms": summary["detector_latency_ms"],
                            "total_frame_latency_ms": np.nan,
                            "unique_track_ids": np.nan,
                            "mean_detection_confidence": summary["mean_detection_confidence"],
                        }
                    )

                comparison_rows.append(row)

            except Exception as exc:
                error_summary = {
                    "model": model_name,
                    "source": str(media_path),
                    "media_type": "unknown",
                    "error": str(exc),
                }
                all_summaries.append(error_summary)
                print(f"ERROR while processing {media_path}: {exc}")

    finally:
        media_progress.close()
        del detector
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


combined_json = SESSION_ROOT / "all_run_summaries.json"
combined_csv = SESSION_ROOT / "all_run_summaries.csv"
comparison_csv = SESSION_ROOT / "runtime_comparison.csv"

combined_json.write_text(json.dumps(all_summaries, indent=2), encoding="utf-8")

flat_rows = []
for item in all_summaries:
    flat_rows.append(item)
pd.json_normalize(flat_rows, sep="__").to_csv(combined_csv, index=False)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(comparison_csv, index=False)

print("\\nBATCH PROCESSING COMPLETE")
print("Session root      :", SESSION_ROOT)
print("Combined JSON     :", combined_json)
print("Combined CSV      :", combined_csv)
print("Comparison CSV    :", comparison_csv)

display(comparison_df.head(50))

\n####################################################################################################
LOADING MODEL: YOLO26s
####################################################################################################
Loading YOLO26s checkpoint:
/content/aerial_final_model_cache/yolo26s_best.pt
YOLO26s checkpoint loaded successfully.
YOLO26s loaded successfully.


YOLO26s batch:   0%|          | 0/11 [00:00<?, ?file/s]

Warming up YOLO26s for 2 run(s)...
Warm-up: COMPLETE


YOLO26s video:   0%|          | 0/1283 [00:00<?, ?frame/s]

Warming up YOLO26s for 2 run(s)...
Warm-up: COMPLETE
Warming up YOLO26s for 2 run(s)...
Warm-up: COMPLETE
Warming up YOLO26s for 2 run(s)...
Warm-up: COMPLETE
Warming up YOLO26s for 2 run(s)...
Warm-up: COMPLETE
Warming up YOLO26s for 2 run(s)...
Warm-up: COMPLETE
Warming up YOLO26s for 2 run(s)...
Warm-up: COMPLETE
Warming up YOLO26s for 2 run(s)...
Warm-up: COMPLETE
Warming up YOLO26s for 2 run(s)...
Warm-up: COMPLETE
Warming up YOLO26s for 2 run(s)...
Warm-up: COMPLETE
Warming up YOLO26s for 2 run(s)...
Warm-up: COMPLETE
\n####################################################################################################
LOADING MODEL: RT-DETR-R18
####################################################################################################
faster-coco-eval: READY
Cloning official RT-DETR repository...
Preparing pinned RT-DETR source...
RT-DETR repository: READY


...

BPD-YOLOn/L-FPN batch:   0%|          | 0/11 [00:00<?, ?file/s]

Warming up BPD-YOLOn/L-FPN for 2 run(s)...
Warm-up: COMPLETE


BPD-YOLOn/L-FPN video:   0%|          | 0/1283 [00:00<?, ?frame/s]

Warming up BPD-YOLOn/L-FPN for 2 run(s)...
Warm-up: COMPLETE
Warming up BPD-YOLOn/L-FPN for 2 run(s)...
Warm-up: COMPLETE
Warming up BPD-YOLOn/L-FPN for 2 run(s)...
Warm-up: COMPLETE
Warming up BPD-YOLOn/L-FPN for 2 run(s)...
Warm-up: COMPLETE
Warming up BPD-YOLOn/L-FPN for 2 run(s)...
Warm-up: COMPLETE
Warming up BPD-YOLOn/L-FPN for 2 run(s)...
Warm-up: COMPLETE
Warming up BPD-YOLOn/L-FPN for 2 run(s)...
Warm-up: COMPLETE
Warming up BPD-YOLOn/L-FPN for 2 run(s)...
Warm-up: COMPLETE
Warming up BPD-YOLOn/L-FPN for 2 run(s)...
Warm-up: COMPLETE
Warming up BPD-YOLOn/L-FPN for 2 run(s)...
Warm-up: COMPLETE
\nBATCH PROCESSING COMPLETE
Session root      : /content/drive/MyDrive/aerial_human_detection/batch_inference_outputs_L4_V2/batch_run_20260814_182555
Combined JSON     : /content/drive/MyDrive/aerial_human_detection/batch_inference_outputs_L4_V2/batch_run_20260814_182555/all_run_summaries.json
Combined CSV      : /content/drive/MyDrive/aerial_human_detection/batch_inference_outputs_L4_V2

,model,source,media_type,step3_val_map50_95,step3_val_ap50,processing_fps,detector_latency_ms,total_frame_latency_ms,unique_track_ids,mean_detection_confidence
0,YOLO26s,/content/drive/MyDrive/referee_videos/1.mp4,video,0.31060,0.67438,19.022555,20.218492,27.749117,5.0,0.513496
1,YOLO26s,/content/drive/MyDrive/referee_videos/v4 - fra...,image,0.31060,0.67438,52.289698,19.124226,NaN,NaN,0.467096
2,YOLO26s,/content/drive/MyDrive/referee_videos/v4 - fra...,image,0.31060,0.67438,53.373069,18.736041,NaN,NaN,0.694980
3,YOLO26s,/content/drive/MyDrive/referee_videos/v4 - fra...,image,0.31060,0.67438,53.162962,18.810088,NaN,NaN,0.232808
4,YOLO26s,/content/drive/MyDrive/referee_videos/vid3 - f...,image,0.31060,0.67438,54.691269,18.284454,NaN,NaN,0.270020
5,YOLO26s,/content/drive/MyDrive/referee_videos/vid3 - f...,image,0.31060,0.67438,53.814168,18.582467,NaN,NaN,0.402493
6,YOLO26s,/content/drive/MyDrive/referee_videos/vid3 - f...,image,0.31060,0.67438,54.665537,18.293061,NaN,NaN,0.130061
7,YOLO26s,/content/drive/MyDrive/referee_videos/vid3 - f...,image,0.31060,0.67438,55.415668,18.045438,NaN,NaN,0.657424
8,YOLO26s,/content/drive/MyDrive/referee_videos/vid3 - f...,image,0.31060,0.67438,54.876237,18.222824,NaN,NaN,0.462398
9,YOLO26s,/content/drive/MyDrive/referee_videos/vid3 - f...,image,0.31060,0.67438,54.412408,18.378161,NaN,NaN,0.324410


## 15. Optional quick summary by model

In [16]:
if not comparison_rows:
    print("No successful runs were recorded.")
else:
    comparison_df = pd.DataFrame(comparison_rows)

    numeric_cols = [
        "processing_fps",
        "detector_latency_ms",
        "total_frame_latency_ms",
        "unique_track_ids",
        "mean_detection_confidence",
        "step3_val_map50_95",
        "step3_val_ap50",
    ]

    available_cols = ["model"] + [c for c in numeric_cols if c in comparison_df.columns]

    summary_df = (
        comparison_df[available_cols]
        .groupby("model", as_index=False)
        .mean(numeric_only=True)
    )

    summary_path = SESSION_ROOT / "mean_summary_by_model.csv"
    summary_df.to_csv(summary_path, index=False)

    print("Mean summary CSV:", summary_path)
    display(summary_df)

Mean summary CSV: /content/drive/MyDrive/aerial_human_detection/batch_inference_outputs_L4_V2/batch_run_20260814_182555/mean_summary_by_model.csv


,model,processing_fps,detector_latency_ms,total_frame_latency_ms,unique_track_ids,mean_detection_confidence,step3_val_map50_95,step3_val_ap50
0,BPD-YOLOn/L-FPN,51.813065,18.317486,23.832899,5.0,0.351313,0.29366,0.65557
1,RT-DETR-R18,28.348715,33.018193,88.307843,23.0,0.154729,0.34854,0.70839
2,YOLO26s,50.875866,18.658578,27.749117,5.0,0.420756,0.31060,0.67438


## Notes

- The notebook supports:
  - a local Drive folder,
  - a shared Drive folder URL,
  - a shared Drive file URL.
- If you use a shared folder, `gdown` will download its contents first.
- Images are processed with detection only.
- Videos are processed with detection + common ByteTrack.
- The same ByteTrack settings are used for all three detector architectures.
- `Mean Det Conf` is a confidence statistic, not ground-truth accuracy.
- `Step3 Val mAP50-95` and `AP50` are fixed reference metrics from Step 3.
- HOTA / IDF1 / MOTA are not reported because no tracking ground truth exists here.